<h1> Predictive Modelling </h1>

This notebook develops predictive models for identifying hepatic steatosis in the NHANES 2017–March 2020 analysis cohort and examines whether adding metabolic phenotype membership as a predictor improves model performance. The training and test cohorts and their metabolic phenotype assignments were produced in `02-metabolic-phenotyping.ipynb`, with phenotype discovery performed using the training cohort only.

Three models are developed:

1. Logistic regression as an interpretable baseline
2. XGBoost using demographic, clinical and metabolic predictors
3. XGBoost using the same predictors with metabolic phenotype added

All models are fitted using the same training participants and generate predictions for the same held-out test participants. Their predicted probabilities and binary classifications are saved for subsequent performance evaluation and interpretation in `04-evaluation-interpretation.ipynb`.

<h2> Importing libraries and defining paths </h2>

The required libraries for data handling, preprocessing, pipeline construction, model development and model storage are imported. A fixed random state of `42` is used to support reproducible model fitting.

Paths are defined for loading the training and test cohorts produced by `02-metabolic-phenotyping.ipynb`. Additional paths are defined for saving the test-set predictions and fitted model pipelines under `../data/processed`. The required output directories are created if they do not already exist.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier

RANDOM_STATE = 42

PROCESSED_DIR = Path("../data/processed")

TRAIN_DATA_PATH = PROCESSED_DIR / "train_clustered.csv"
TEST_DATA_PATH = PROCESSED_DIR / "test_clustered.csv"
PREDICTIONS_PATH = PROCESSED_DIR / "model_predictions.csv"
MODEL_DIR = PROCESSED_DIR / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

<h2> Loading and validating the training and test cohorts </h2>

The training and test cohorts produced by `02-metabolic-phenotyping.ipynb` are loaded from `train_clustered.csv` and `test_clustered.csv`, respectively. Both datasets contain the selected demographic, clinical and metabolic predictors, hepatic steatosis outcome, numerical cluster assignment and descriptive metabolic phenotype for each participant.

The number of participants and the distributions of the hepatic steatosis outcome and metabolic phenotypes are displayed separately for the training and test cohorts. The first five rows of the training and test cohorts are also displayed to verify their structure and contents.

In [2]:
train_df = pd.read_csv(TRAIN_DATA_PATH)
test_df = pd.read_csv(TEST_DATA_PATH)

required_columns = [
    "SEQN",
    "age",
    "sex",
    "race_ethnicity",
    "bmi",
    "waist_cm",
    "systolic_bp",
    "diastolic_bp",
    "hdl_mg_dl",
    "triglycerides_mg_dl",
    "glucose_mg_dl",
    "diabetes_status",
    "cap_db_m",
    "hepatic_steatosis",
    "cluster",
    "metabolic_phenotype",
]

expected_phenotypes = {
    "Lower metabolic-risk",
    "Higher metabolic-risk",
}

datasets = {
    "training": train_df,
    "test": test_df,
}

for name, dataset in datasets.items():
    missing_columns = [
        column for column in required_columns
        if column not in dataset.columns
    ]

    assert not missing_columns, (
        f"Missing columns in {name} dataset: {missing_columns}"
    )
    assert dataset["SEQN"].is_unique
    assert dataset[required_columns].isna().sum().sum() == 0
    assert set(dataset["hepatic_steatosis"].unique()) == {0, 1}
    assert set(dataset["cluster"].unique()) == {0, 1}
    assert set(dataset["metabolic_phenotype"].unique()) == expected_phenotypes

# Confirm that training and test participants are separate
assert set(train_df["SEQN"]).isdisjoint(set(test_df["SEQN"]))
assert len(train_df) + len(test_df) == 3270

print(f"Training participants: {len(train_df):,}")
print(f"Test participants:     {len(test_df):,}")
print(f"Combined participants: {len(train_df) + len(test_df):,}")

print("\nTraining outcome distribution:")
print(train_df["hepatic_steatosis"].value_counts().sort_index())

print("\nTest outcome distribution:")
print(test_df["hepatic_steatosis"].value_counts().sort_index())

print("\nTraining metabolic phenotypes:")
print(train_df["metabolic_phenotype"].value_counts())

print("\nTest metabolic phenotypes:")
print(test_df["metabolic_phenotype"].value_counts())

print("\nFirst five training records:")
display(train_df.head())

print("\nFirst five test records:")
display(test_df.head())

Training participants: 2,616
Test participants:     654
Combined participants: 3,270

Training outcome distribution:
hepatic_steatosis
0    1110
1    1506
Name: count, dtype: int64

Test outcome distribution:
hepatic_steatosis
0    278
1    376
Name: count, dtype: int64

Training metabolic phenotypes:
metabolic_phenotype
Lower metabolic-risk     1463
Higher metabolic-risk    1153
Name: count, dtype: int64

Test metabolic phenotypes:
metabolic_phenotype
Lower metabolic-risk     357
Higher metabolic-risk    297
Name: count, dtype: int64

First five training records:


,SEQN,age,sex,race_ethnicity,bmi,waist_cm,systolic_bp,diastolic_bp,hdl_mg_dl,triglycerides_mg_dl,glucose_mg_dl,diabetes_status,cap_db_m,hepatic_steatosis,cluster,metabolic_phenotype
0,110604.0,19.0,Male,Non-Hispanic White,21.9,79.0,99.666667,66.333333,41.0,55.0,97.0,No,228.0,0,0,Lower metabolic-risk
1,117697.0,19.0,Female,Other Hispanic,24.5,78.6,104.666667,67.666667,62.0,48.0,87.0,No,211.0,0,0,Lower metabolic-risk
2,124665.0,65.0,Male,Non-Hispanic Black,33.7,116.4,143.666667,82.666667,38.0,133.0,128.0,No,373.0,1,1,Higher metabolic-risk
3,111297.0,70.0,Male,Non-Hispanic White,32.4,116.4,120.666667,73.333333,38.0,156.0,107.0,No,268.0,1,1,Higher metabolic-risk
4,121483.0,18.0,Female,Non-Hispanic White,23.3,81.8,106.333333,65.666667,51.0,49.0,104.0,No,196.0,0,0,Lower metabolic-risk



First five test records:


,SEQN,age,sex,race_ethnicity,bmi,waist_cm,systolic_bp,diastolic_bp,hdl_mg_dl,triglycerides_mg_dl,glucose_mg_dl,diabetes_status,cap_db_m,hepatic_steatosis,cluster,metabolic_phenotype
0,121606.0,44.0,Female,Mexican American,27.6,98.0,108.333333,59.000000,62.0,72.0,92.0,No,287.0,1,0,Lower metabolic-risk
1,120403.0,51.0,Female,Non-Hispanic Black,42.4,124.2,120.333333,80.666667,38.0,74.0,117.0,No,324.0,1,1,Higher metabolic-risk
2,115908.0,65.0,Male,Non-Hispanic Black,20.6,77.7,113.333333,66.000000,76.0,24.0,127.0,No,231.0,0,0,Lower metabolic-risk
3,119958.0,30.0,Male,Non-Hispanic White,44.9,134.9,125.666667,82.000000,37.0,134.0,105.0,No,375.0,1,1,Higher metabolic-risk
4,114423.0,63.0,Female,Non-Hispanic Asian,21.5,81.0,133.333333,74.666667,45.0,129.0,153.0,Yes,332.0,1,0,Lower metabolic-risk


The validation checks confirm that both datasets contain all expected columns, unique participant identifiers (`SEQN`), no missing values in the required variables, both hepatic steatosis outcome classes, both cluster labels and both metabolic phenotype categories. The training and test cohorts do not contain overlapping participants and together contain all 3,270 participants.

<h2> Defining the outcome and predictor sets </h2>

The binary outcome is `hepatic_steatosis`, where `1` and `0` represent the presence and absence of hepatic steatosis, respectively.

The `base_features` set contains eight numerical and three categorical variables representing demographic, anthropometric, clinical and metabolic information.

A second predictor set, `features_with_phenotype`, adds `metabolic_phenotype`, a binary categorical variable indicating membership in the lower or higher metabolic-risk phenotype. These phenotypes were identified through K-means clustering of the training cohort in `02-metabolic-phenotyping.ipynb`. Test participants were subsequently assigned to the learned phenotypes without refitting the scaler or clustering model. CAP measurements and the `hepatic_steatosis` outcome were not used to derive the phenotypes.

`base_features` is used for logistic regression and XGBoost without phenotype information, while `features_with_phenotype` is used for the second XGBoost model, ensuring that the two XGBoost models differ only in whether phenotype membership is included as a predictor.

`SEQN` is excluded because it is a participant identifier, while `cap_db_m` is excluded to prevent outcome leakage because the hepatic steatosis outcome is derived directly from CAP. The numerical `cluster` variable is also excluded because it contains the same phenotype membership information as `metabolic_phenotype`.

In [3]:
outcome = "hepatic_steatosis"

numeric_features = [
    "age",
    "bmi",
    "waist_cm",
    "systolic_bp",
    "diastolic_bp",
    "hdl_mg_dl",
    "triglycerides_mg_dl",
    "glucose_mg_dl",
] 

categorical_features = [
    "sex",
    "race_ethnicity",
    "diabetes_status",
]

base_features = numeric_features + categorical_features

phenotype_feature = ["metabolic_phenotype"]
features_with_phenotype = base_features + phenotype_feature

print(f"Base predictors:             {len(base_features)}")
print(f"Predictors with phenotype:   {len(features_with_phenotype)}")

print("\nBase predictors:")
for feature in base_features:
    print(f"- {feature}")

print("\nAdditional phenotype predictor:")
print(f"- {phenotype_feature[0]}")

Base predictors:             11
Predictors with phenotype:   12

Base predictors:
- age
- bmi
- waist_cm
- systolic_bp
- diastolic_bp
- hdl_mg_dl
- triglycerides_mg_dl
- glucose_mg_dl
- sex
- race_ethnicity
- diabetes_status

Additional phenotype predictor:
- metabolic_phenotype


<h2> Separating predictors and outcome </h2>

The training and test cohorts are separated into predictor datasets (`X`) and hepatic steatosis outcome variables (`y_train` and `y_test`).

Two versions of the predictor data are prepared. The base datasets (`X_train_base` and `X_test_base`) contain the 11 demographic, clinical and metabolic predictors, while the phenotype-enhanced datasets (`X_train_with_phenotype` and `X_test_with_phenotype`) additionally contain the `metabolic_phenotype` predictor.

The dimensions of each predictor and outcome dataset are displayed. 

In [4]:
# Prepare modelling datasets

# Base predictor datasets
X_train_base = train_df[base_features].copy()
X_test_base = test_df[base_features].copy()

# Predictor datasets including metabolic phenotype
X_train_with_phenotype = train_df[features_with_phenotype].copy()
X_test_with_phenotype = test_df[features_with_phenotype].copy()

# Outcome
y_train = train_df[outcome].copy()
y_test = test_df[outcome].copy()

# Validate predictor-outcome alignment
assert X_train_base.index.equals(y_train.index)
assert X_test_base.index.equals(y_test.index)
assert X_train_with_phenotype.index.equals(y_train.index)
assert X_test_with_phenotype.index.equals(y_test.index)

print(f"X_train_base:             {X_train_base.shape}")
print(f"X_test_base:              {X_test_base.shape}")
print(f"X_train_with_phenotype:   {X_train_with_phenotype.shape}")
print(f"X_test_with_phenotype:    {X_test_with_phenotype.shape}")
print(f"y_train:                  {y_train.shape}")
print(f"y_test:                   {y_test.shape}")

X_train_base:             (2616, 11)
X_test_base:              (654, 11)
X_train_with_phenotype:   (2616, 12)
X_test_with_phenotype:    (654, 12)
y_train:                  (2616,)
y_test:                   (654,)


The validation checks confirm that the rows in both the base and phenotype-enhanced predictor datasets remain aligned with their corresponding outcome values.

<h2> Defining preprocessing pipelines </h2>

For logistic regression, numerical predictors are standardised using `StandardScaler` because its coefficients are sensitive to differences in predictor scale. Categorical predictors are converted into numerical indicator variables using `OneHotEncoder`. The `Female`, `Non-Hispanic White` and `No` categories are explicitly omitted as the reference categories for `sex`, `race_ethnicity` and `diabetes_status` variables, respectively.

For XGBoost, numerical predictors are retained in their original units because tree-based models do not require standardisation. Categorical predictors are one-hot encoded, with all categories retained. The phenotype-enhanced pipeline additionally includes `metabolic_phenotype` as a categorical predictor.

At this stage, the preprocessing objects are defined but not yet fitted. They are subsequently incorporated into each model pipeline and fitted using only the training data, preventing information from the test set from influencing preprocessing. 

`handle_unknown="ignore"` is specified as a safeguard so that categories present in the test or future validation data but absent from the training cohort do not cause an encoding error.

In [5]:
# Logistic regression:
# standardise numerical predictors and one-hot encode categorical predictors
logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop=[
                    "Female",
                    "Non-Hispanic White",
                    "No",
                ],
            ),
            categorical_features,
        ),
    ],
    remainder="drop",
)

# XGBoost without phenotype:
# retain numerical values and one-hot encode categorical predictors
xgb_base_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ],
    remainder="drop",
)

# XGBoost with phenotype:
# add metabolic phenotype as another categorical predictor
xgb_phenotype_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features + phenotype_feature,
        ),
    ],
    remainder="drop",
)

print("Preprocessing pipelines defined successfully.")

Preprocessing pipelines defined successfully.


<h2> Developing the predictive models </h2>

<h3> Logistic regression baseline </h3>

A pipeline combines the logistic-regression preprocessor with a multivariable logistic regression classifier. When the pipeline is fitted, the preprocessing steps are first fitted and applied to the training predictors, after which the classifier is fitted using the transformed training predictors and corresponding training outcomes. 

Logistic regression provides a comparatively simple and interpretable baseline for comparison with the XGBoost models. The maximum number of iterations is set to `2,000` to allow the optimisation process sufficient opportunity to converge.

For each test participant, the fitted pipeline generates an estimated probability of hepatic steatosis. Probabilities at or above `0.50` are converted to a binary prediction of `1`, indicating predicted hepatic steatosis, while lower probabilities are classified as `0`, indicating predicted absence of hepatic steatosis. 

In [6]:
logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            logistic_preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_model.fit(X_train_base, y_train)

# Predicted probability of hepatic steatosis
logistic_probability = logistic_model.predict_proba(
    X_test_base
)[:, 1]

# Binary prediction using the conventional 0.50 threshold
logistic_prediction = (
    logistic_probability >= 0.50
).astype(int)

assert len(logistic_probability) == len(y_test)
assert np.all(
    (logistic_probability >= 0)
    & (logistic_probability <= 1)
)

print("Logistic regression fitted successfully.")
print(f"Test predictions generated: {len(logistic_prediction):,}")
print(
    "Predicted steatosis cases at threshold 0.50: "
    f"{logistic_prediction.sum():,}"
)

Logistic regression fitted successfully.
Test predictions generated: 654
Predicted steatosis cases at threshold 0.50: 401


The model generated predictions for all 654 test participants, including 401 predictions of hepatic steatosis at the `0.50` probability threshold. Both outcome classes were predicted, and the absence of a convergence warning indicates that model fitting completed normally. The validation checks also confirm that all predicted probabilities fall between zero and one.

The number of predicted cases does not establish model performance. The predictions are compared with the true test outcomes using AUROC, AUPRC, sensitivity, specificity and other metrics in `04-evaluation-interpretation.ipynb`.

<h3> XGBoost without metabolic phenotype </h3>

A second pipeline combines the base XGBoost preprocessor with an `XGBClassifier`. This model uses the same eleven demographic, clinical and metabolic predictors as the logistic regression model but can capture nonlinear relationships and interactions between predictors.

A moderate-complexity configuration is used. It comprises `300` trees, a learning rate of `0.05`, a maximum tree depth of `3`, and 80% row and feature subsampling. A fixed random state supports reproducibility, while a binary logistic objective is used to estimate the probability of hepatic steatosis, with log loss specified as an evaluation metric.

The fitted pipeline generates predicted probabilities for the test cohort. Probabilities at or above `0.50` are classified as predicted hepatic steatosis. The same XGBoost settings and train/test split are subsequently used for the phenotype-enhanced model, ensuring that the inclusion of metabolic phenotype is the only deliberate difference between the two XGBoost models.

In [7]:
# Shared XGBoost settings for a fair comparison between models
xgb_parameters = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 3,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

xgb_base_model = Pipeline(
    steps=[
        (
            "preprocessor",
            xgb_base_preprocessor,
        ),
        (
            "classifier",
            XGBClassifier(**xgb_parameters),
        ),
    ]
)

xgb_base_model.fit(X_train_base, y_train)

# Generate test-set predictions
xgb_base_probability = xgb_base_model.predict_proba(
    X_test_base
)[:, 1]

xgb_base_prediction = (
    xgb_base_probability >= 0.50
).astype(int)

assert len(xgb_base_probability) == len(y_test)
assert np.all(
    (xgb_base_probability >= 0)
    & (xgb_base_probability <= 1)
)

print("XGBoost without phenotype fitted successfully.")
print(f"Test predictions generated: {len(xgb_base_prediction):,}")
print(
    "Predicted steatosis cases at threshold 0.50: "
    f"{xgb_base_prediction.sum():,}"
)

XGBoost without phenotype fitted successfully.
Test predictions generated: 654
Predicted steatosis cases at threshold 0.50: 397


The model generated predictions for all 654 test participants, including 397 predictions of hepatic steatosis at the `0.50` probability threshold. The validation checks confirm that all predicted probabilities fall between zero and one.

XGBoost without phenotype information predicted 397 positive cases, compared with 401 from logistic regression. The difference in total prediction counts does not indicate which model performs better, and the models may also differ in which participants they classify as positive and in the probabilities they assign.

<h3> XGBoost with metabolic phenotype </h3>

A third pipeline combines the phenotype-enhanced preprocessor with an `XGBClassifier`. It uses the same demographic, clinical and metabolic predictors and XGBoost settings as the preceding model, with `metabolic_phenotype` added as the only additional predictor.

Because all other model settings and the train/test split remain unchanged, this controlled comparison examines whether explicit information about data-driven phenotype membership contributes predictive value beyond the individual metabolic variables from which the phenotypes were derived.

The fitted pipeline generates predicted probabilities and binary classifications for the same test participants using the `0.50` probability threshold.

In [8]:
# Train XGBoost with metabolic phenotype 

xgb_phenotype_model = Pipeline(
    steps=[
        (
            "preprocessor",
            xgb_phenotype_preprocessor,
        ),
        (
            "classifier",
            XGBClassifier(**xgb_parameters),
        ),
    ]
)

xgb_phenotype_model.fit(
    X_train_with_phenotype,
    y_train,
)

# Generate test-set predictions
xgb_phenotype_probability = (
    xgb_phenotype_model.predict_proba(
        X_test_with_phenotype
    )[:, 1]
)

xgb_phenotype_prediction = (
    xgb_phenotype_probability >= 0.50
).astype(int)

assert len(xgb_phenotype_probability) == len(y_test)
assert np.all(
    (xgb_phenotype_probability >= 0)
    & (xgb_phenotype_probability <= 1)
)

print("XGBoost with phenotype fitted successfully.")
print(
    f"Test predictions generated: "
    f"{len(xgb_phenotype_prediction):,}"
)
print(
    "Predicted steatosis cases at threshold 0.50: "
    f"{xgb_phenotype_prediction.sum():,}"
)

XGBoost with phenotype fitted successfully.
Test predictions generated: 654
Predicted steatosis cases at threshold 0.50: 410


The phenotype-enhanced model generated predictions for all 654 test participants, including 410 predictions of hepatic steatosis at the `0.50` probability threshold, compared with 397 predictions from XGBoost without phenotype information. The validation checks confirm that all predicted probabilities fall between zero and one.

The difference in prediction counts alone does not indicate which model performs better. Performance is determined by comparing the predicted probabilities and classifications with the true test outcomes in `04-evaluation-interpretation.ipynb`.

<h2> Saving model outputs </h2>

<h3> Saving test-set predictions </h3>

A prediction table is created for the held-out test cohort. It contains each participant's `SEQN`, true hepatic steatosis outcome, numerical cluster assignment and descriptive metabolic phenotype.

The predicted probabilities and binary classifications generated by logistic regression, XGBoost without phenotype information and XGBoost with phenotype information are added to the same table. This provides the information required for model comparison and phenotype-specific evaluation in `04-evaluation-interpretation.ipynb`.

Validation checks are performed before the table is saved locally as `model_predictions.csv`.

In [9]:
predictions_df = test_df[
    [
        "SEQN",
        outcome,
        "cluster",
        "metabolic_phenotype",
    ]
].copy()

predictions_df["logistic_probability"] = (
    logistic_probability
)
predictions_df["logistic_prediction"] = (
    logistic_prediction
)

predictions_df["xgb_base_probability"] = (
    xgb_base_probability
)
predictions_df["xgb_base_prediction"] = (
    xgb_base_prediction
)

predictions_df["xgb_phenotype_probability"] = (
    xgb_phenotype_probability
)
predictions_df["xgb_phenotype_prediction"] = (
    xgb_phenotype_prediction
)

assert len(predictions_df) == len(test_df)
assert predictions_df["SEQN"].is_unique
assert predictions_df.isna().sum().sum() == 0

predictions_df.to_csv(
    PREDICTIONS_PATH,
    index=False,
)

print(f"Prediction records: {len(predictions_df):,}")
print(f"Saved predictions to: {PREDICTIONS_PATH}")

predictions_df.head()

Prediction records: 654
Saved predictions to: ../data/processed/model_predictions.csv


,SEQN,hepatic_steatosis,cluster,metabolic_phenotype,logistic_probability,logistic_prediction,xgb_base_probability,xgb_base_prediction,xgb_phenotype_probability,xgb_phenotype_prediction
0,121606.0,1,0,Lower metabolic-risk,0.440789,0,0.548354,1,0.549772,1
1,120403.0,1,1,Higher metabolic-risk,0.864101,1,0.941364,1,0.940724,1
2,115908.0,0,0,Lower metabolic-risk,0.113390,0,0.339191,0,0.376669,0
3,119958.0,1,1,Higher metabolic-risk,0.969329,1,0.956138,1,0.939116,1
4,114423.0,1,0,Lower metabolic-risk,0.550238,1,0.776224,1,0.822910,1


The validation checks confirm that the table contains one record for each of the 654 test participants, every `SEQN` is unique and no values are missing. 

The prediction table was saved successfully under `../data/processed`. The displayed first five rows provide a preliminary check of its structure and contents.

<h3> Saving the fitted model pipelines </h3>

The three fitted pipelines are saved using `joblib`. Each saved object contains both the fitted preprocessing steps and the corresponding classifier, ensuring that the same training-derived transformations are applied when the model is loaded for subsequent analysis.

Saving the complete pipelines allows `04-evaluation-interpretation.ipynb` to load the fitted models for evaluation and interpretation without repeating model training. The model files are stored locally under `../data/processed/models`, which is excluded from Git.

In [10]:
model_paths = {
    "logistic_regression": (
        MODEL_DIR / "logistic_regression.joblib"
    ),
    "xgboost_without_phenotype": (
        MODEL_DIR / "xgboost_without_phenotype.joblib"
    ),
    "xgboost_with_phenotype": (
        MODEL_DIR / "xgboost_with_phenotype.joblib"
    ),
}

joblib.dump(
    logistic_model,
    model_paths["logistic_regression"],
)

joblib.dump(
    xgb_base_model,
    model_paths["xgboost_without_phenotype"],
)

joblib.dump(
    xgb_phenotype_model,
    model_paths["xgboost_with_phenotype"],
)

print("Saved fitted model pipelines:")
for name, path in model_paths.items():
    print(f"- {name}: {path}")

Saved fitted model pipelines:
- logistic_regression: ../data/processed/models/logistic_regression.joblib
- xgboost_without_phenotype: ../data/processed/models/xgboost_without_phenotype.joblib
- xgboost_with_phenotype: ../data/processed/models/xgboost_with_phenotype.joblib


The logistic regression pipeline and both XGBoost pipelines were saved successfully in the designated model directory.

<h2> Validating the saved outputs </h2>

The saved prediction table is reloaded and checked to confirm that it contains the expected columns in the correct order, contains exactly the same 654 participants as the test cohort, includes unique participant identifiers and has no missing values.

The predicted probabilities are also checked to ensure that they fall between zero and one, while the binary classifications are checked to ensure that they contain only `0` and `1`. Each saved model file is also checked to confirm that it exists and is not empty.

In [11]:
expected_prediction_columns = [
    "SEQN",
    "hepatic_steatosis",
    "cluster",
    "metabolic_phenotype",
    "logistic_probability",
    "logistic_prediction",
    "xgb_base_probability",
    "xgb_base_prediction",
    "xgb_phenotype_probability",
    "xgb_phenotype_prediction",
]

saved_predictions = pd.read_csv(PREDICTIONS_PATH)

assert list(saved_predictions.columns) == expected_prediction_columns
assert len(saved_predictions) == len(test_df)
assert saved_predictions["SEQN"].is_unique
assert saved_predictions.isna().sum().sum() == 0
assert set(saved_predictions["SEQN"]) == set(test_df["SEQN"])

probability_columns = [
    "logistic_probability",
    "xgb_base_probability",
    "xgb_phenotype_probability",
]

prediction_columns = [
    "logistic_prediction",
    "xgb_base_prediction",
    "xgb_phenotype_prediction",
]

for column in probability_columns:
    assert saved_predictions[column].between(0, 1).all()

for column in prediction_columns:
    assert set(saved_predictions[column].unique()).issubset({0, 1})

for path in model_paths.values():
    assert path.exists()
    assert path.stat().st_size > 0

print("Final validation checks passed.")
print(f"Saved prediction records: {len(saved_predictions):,}")
print(f"Saved fitted models:       {len(model_paths)}")

Final validation checks passed.
Saved prediction records: 654
Saved fitted models:       3


The final validation checks passed successfully. The saved outputs contain 654 test-set prediction records and three fitted model pipelines for subsequent evaluation and interpretation.